# Chapter 20 — Synthetic Priors

Reproduces:
- Figure 20.1: Episode samples from each of the four prior families (PCA projection).
- Figure 20.2: Per-prior label distributions.
- Figure 20.3: Downstream ICL accuracy of a small PFN trained on each prior, evaluated cross-prior.


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from tabkernels.priors import (
    SCMPrior, SCMConfig, ARFPrior, MLPSCMPrior, KnowledgeGraphPrior,
)
from tabkernels.training import PFNTrainer

torch.manual_seed(0); np.random.seed(0)
_p = os.getcwd()
while _p and not os.path.isdir(os.path.join(_p, 'affinity', 'book')):
    _p = os.path.dirname(_p)
FIGURES_DIR = os.path.join(_p, 'affinity', 'book', 'figures')
os.makedirs(FIGURES_DIR, exist_ok=True)


## Construct the four priors

The CH20 priors are: **SCM** (random DAG with linear or MLP equations), **ARF**
(data-driven via leaf partition over a seed corpus), **MLP-SCM** (ARF features
with random-MLP labels — our hybrid), and **KG** (knowledge-graph-augmented
sketch with concept embeddings).


In [ ]:
# SCM: structural causal model with MLP equations.
scm = SCMPrior(SCMConfig(structural='mlp', mlp_hidden=12, edge_prob=0.5,
                         noise_scale=0.3))

# ARF: needs a seed corpus. Use a Gaussian mixture as a stand-in for a real
# OpenML corpus; this keeps the demo self-contained.
torch.manual_seed(7)
n_corpus = 600; D = 4
mode_assign = torch.randint(3, (n_corpus,))
mode_centres = torch.tensor([[2.0, 0.0, 0.0, 0.0],
                              [0.0, 2.0, -2.0, 0.0],
                              [-2.0, -2.0, 1.0, 1.0]])
seed_corpus = mode_centres[mode_assign] + 0.5 * torch.randn(n_corpus, D)
arf = ARFPrior(corpus=seed_corpus, max_leaves=24)

# MLP-SCM hybrid: ARF features, random-MLP labels.
mlpscm = MLPSCMPrior(feature_prior=arf, hidden=12, depth=2,
                     activation='tanh', noise_scale=0.1)

# KG-augmented prior.
kg = KnowledgeGraphPrior(n_concepts=24, d_embed=8, noise_scale=0.1)

priors = {'SCM': scm, 'ARF': arf, 'MLP-SCM': mlpscm, 'KG': kg}
print(f'priors: {list(priors)}')


## Figure 20.1: Episode samples (PCA projection)

For each prior we sample 400 points and project to 2D. Geometry that matters:
SCM tends to produce elongated correlated clouds (linear or MLP equations);
ARF inherits the corpus's mixture structure; MLP-SCM has the same ARF features
but with structurally different labels; KG induces clusters tied to the
underlying concept hierarchy.


In [ ]:
def sample_features(prior, n=400, d=4, seed_offset=0):
    Xs, ys = [], []
    for k in range(n // 32):
        X_c, y_c, X_q, y_q = prior.sample_episode(32, 0, d=d, seed=seed_offset + k)
        Xs.append(X_c); ys.append(y_c)
    return torch.cat(Xs), torch.cat(ys)


def pca_2d(X):
    Xc = X - X.mean(0)
    U, S, Vh = torch.linalg.svd(Xc, full_matrices=False)
    return (Xc @ Vh[:2].T).numpy()


fig, axes = plt.subplots(1, 4, figsize=(16, 3.8))
for ax, (name, p) in zip(axes, priors.items()):
    X, y = sample_features(p, n=384, d=D, seed_offset=name.__hash__() % 1000)
    P = pca_2d(X)
    sc = ax.scatter(P[:, 0], P[:, 1], c=y.numpy(), cmap='coolwarm', s=14, alpha=0.7)
    ax.set_title(f'{name} (n=384, PCA-2D)'); ax.set_xticks([]); ax.set_yticks([])
    plt.colorbar(sc, ax=ax, fraction=0.046, pad=0.04)
plt.suptitle('Figure 20.1: Episode samples from four synthetic priors')
plt.tight_layout(); plt.savefig(f'{FIGURES_DIR}/fig_20_01_episodes.pdf', bbox_inches='tight')
plt.show()


## Figure 20.2: Label distributions

The four priors generate qualitatively different label distributions: continuous
SCM linear combinations, ARF's concept-conditional values, MLP-SCM's smooth
nonlinear targets, and KG's discretised depth-coded labels.


In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 3))
for ax, (name, p) in zip(axes, priors.items()):
    _, y = sample_features(p, n=512, d=D, seed_offset=42)
    ax.hist(y.numpy(), bins=30, color='C0', alpha=0.85, edgecolor='k')
    ax.set_title(f'{name}'); ax.set_xlabel('y'); ax.set_yticks([])
plt.suptitle('Figure 20.2: Label distributions across priors')
plt.tight_layout(); plt.savefig(f'{FIGURES_DIR}/fig_20_02_label_dists.pdf', bbox_inches='tight')
plt.show()


## Figure 20.3: Cross-prior PFN evaluation

For each prior, we (a) train a small PFN on episodes from that prior, and (b)
evaluate the trained PFN on held-out episodes from *all four priors*. The
diagonal of the resulting matrix is the within-prior performance; off-diagonal
entries quantify generalisation across prior families.

Take-away: a PFN trained on one synthetic prior is not automatically useful on
a different one; the prior is the inductive bias.


In [ ]:
class TinyPFN(nn.Module):
    def __init__(self, d_in, d_emb=48, n_heads=4, n_layers=2, d_ff=96):
        super().__init__()
        self.x_embed = nn.Linear(d_in, d_emb)
        self.y_embed = nn.Linear(1, d_emb)
        self.q_marker = nn.Parameter(torch.randn(d_emb) * 0.02)
        self.layers = nn.ModuleList([
            nn.ModuleDict({
                'ln1': nn.LayerNorm(d_emb),
                'attn': nn.MultiheadAttention(d_emb, n_heads, batch_first=True),
                'ln2': nn.LayerNorm(d_emb),
                'mlp': nn.Sequential(nn.Linear(d_emb, d_ff), nn.GELU(),
                                     nn.Linear(d_ff, d_emb)),
            }) for _ in range(n_layers)
        ])
        self.out_ln = nn.LayerNorm(d_emb)
        self.head = nn.Linear(d_emb, 1)

    def forward(self, X_q, X_ctx, y_ctx):
        ctx = self.x_embed(X_ctx) + self.y_embed(y_ctx.unsqueeze(-1))
        q = self.x_embed(X_q) + self.q_marker
        seq = torch.cat([ctx, q], dim=0).unsqueeze(0)
        nc = X_ctx.shape[0]
        for layer in self.layers:
            n = layer['ln1'](seq)
            a, _ = layer['attn'](n, n, n, need_weights=False)
            seq = seq + a
            seq = seq + layer['mlp'](layer['ln2'](seq))
        return self.head(self.out_ln(seq[0, nc:]))


def standardise_y(y_ctx, y_q):
    """Per-task y standardisation. ICL across heterogeneous priors needs
    consistent target scaling, otherwise a model trained on one prior's range
    can't be evaluated on another."""
    mu = y_ctx.mean(); sd = y_ctx.std().clamp_min(1e-3)
    return (y_ctx - mu) / sd, (y_q - mu) / sd


class StdWrap(nn.Module):
    """Standardisation wrapper that exposes the PFNTrainer call signature."""
    def __init__(self, inner): super().__init__(); self.inner = inner
    def forward(self, X_q, X_ctx, y_ctx):
        mu = y_ctx.mean(); sd = y_ctx.std().clamp_min(1e-3)
        y_norm = (y_ctx - mu) / sd
        out = self.inner(X_q, X_ctx, y_norm).squeeze(-1)
        return out * sd + mu


# Train one PFN per prior. Use the same hyperparameters across priors.
N_STEPS = 500
trained = {}
for name, p in priors.items():
    torch.manual_seed(123)
    inner = TinyPFN(d_in=D, d_emb=48, n_heads=4, n_layers=2, d_ff=96)
    model = StdWrap(inner)
    trainer = PFNTrainer(prior=p, model=model, n_steps=N_STEPS, n_ctx=48,
                         n_query=24, d=D, lr=3e-3, seed=name.__hash__() % 1000)
    state = trainer.train()
    trained[name] = (model, state)
    print(f'  trained PFN on {name:<8s}: final train loss = {state.losses[-1]:.4f}')


In [ ]:
def evaluate_cross(model, eval_prior, n_eval_tasks=20, n_ctx=48, n_q=24):
    model.eval()
    losses = []
    with torch.no_grad():
        for k in range(n_eval_tasks):
            X_c, y_c, X_q, y_q = eval_prior.sample_episode(n_ctx, n_q, D, seed=98765 + k)
            yhat = model(X_q, X_c, y_c)
            # Normalised MSE: divide by Var(y_q) so cross-prior comparisons are fair.
            v = y_q.var().clamp_min(1e-3)
            losses.append((((yhat - y_q) ** 2).mean() / v).item())
    return float(np.mean(losses))


names = list(priors.keys())
mat = np.zeros((len(names), len(names)))
for i, n_train in enumerate(names):
    model, _ = trained[n_train]
    for j, n_eval in enumerate(names):
        mat[i, j] = evaluate_cross(model, priors[n_eval])
print('Cross-prior normalised MSE (rows: train, cols: eval):')
print('             ' + ' '.join(f'{n:>10s}' for n in names))
for i, n in enumerate(names):
    print(f'  {n:>10s} ' + ' '.join(f'{mat[i, j]:10.3f}' for j in range(len(names))))

fig, ax = plt.subplots(1, 1, figsize=(5.5, 5.0))
im = ax.imshow(mat, cmap='viridis', aspect='equal')
for i in range(len(names)):
    for j in range(len(names)):
        ax.text(j, i, f'{mat[i, j]:.2f}', ha='center', va='center',
                color='white' if mat[i, j] > mat.mean() else 'black', fontsize=9)
ax.set_xticks(range(len(names))); ax.set_yticks(range(len(names)))
ax.set_xticklabels(names); ax.set_yticklabels(names)
ax.set_xlabel('eval prior'); ax.set_ylabel('train prior')
ax.set_title('Figure 20.3: Cross-prior normalised MSE\n(diagonal = within-prior; lower is better)')
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout(); plt.savefig(f'{FIGURES_DIR}/fig_20_03_cross_prior.pdf', bbox_inches='tight')
plt.show()
